# 03 — DistilBERT vs PubMedBERT fine-tuning

Run from the project root:

```bash
python -m src.transformer_models --model distilbert-base-uncased --task B
python -m src.transformer_models --model microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext --task B
```

This notebook reads each `models/*/metrics.json` and produces a single
comparison table + bar chart so the biomedical-pretrained-advantage
hypothesis can be inspected at a glance.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('../models')
rows = []
for d in ROOT.iterdir():
    m = d / 'metrics.json'
    if m.exists():
        payload = json.loads(m.read_text())
        rows.append({
            'model': payload.get('model', d.name),
            'task': payload.get('task'),
            'val_macro_f1': payload.get('val', {}).get('macro_f1'),
            'test_macro_f1': payload.get('test', {}).get('macro_f1'),
            'val_mae': payload.get('val', {}).get('mae'),
            'test_mae': payload.get('test', {}).get('mae'),
        })
comp = pd.DataFrame(rows)
comp

In [ ]:
# include the TF-IDF baseline
import json
baseline = json.loads(Path('../results/baseline_B_metrics.json').read_text())
baseline_row = pd.DataFrame([{ 
    'model': 'TF-IDF + LogReg',
    'task': 'B',
    'val_macro_f1': baseline['val']['macro_f1'],
    'test_macro_f1': baseline['test']['macro_f1'],
}])
all_rows = pd.concat([baseline_row, comp], ignore_index=True)
ax = all_rows.set_index('model')[['val_macro_f1', 'test_macro_f1']].plot.barh(figsize=(8, 4))
ax.set_xlabel('Macro-F1'); ax.set_title('Task B model comparison')